# Apache Iceberg Features: Ingestion Inspection, DML, Time Travel & Evolution in PySpark

This notebook demonstrates key capabilities of Apache Iceberg tables using PySpark and Spark SQL:
1. **Table Ingestion Inspection**: View commit history, snapshots, manifests, and data/metadata files.
2. **DML Demonstration**: Perform direct `INSERT`, `UPDATE`, and `DELETE` operations on the Iceberg table using Spark SQL.
3. **Time Travel**: Query historical table states using specific snapshot IDs and epoch timestamps.
4. **Schema Evolution**: Alter table schemas dynamically (adding, renaming, updating, and dropping columns) without rewriting the table.
5. **Partition Evolution**: Evolve partitioning layout schemas dynamically in-place without losing history or rewriting old data files.

## Step 1: Initialize Spark Session with Apache Iceberg Support

We stop the default pre-created Spark session and re-initialize it explicitly. We configure Spark to load the Iceberg Spark runtime packages via `spark.jars.packages` and map our Hadoop Catalog (`gcs_hadoop_catalog`) to the GCS staging bucket warehouse path.

In [ ]:
import os
from pyspark.sql import SparkSession

# REPLACE WITH YOUR GCS STAGING BUCKET NAME
WAREHOUSE_PATH = "gs://YOUR_STAGING_BUCKET/warehouse"

# Stop pre-existing default Spark session before starting our custom configured one
try:
    spark.stop()
    print("Stopped active SparkSession.")
except:
    pass

spark = SparkSession.builder \
    .appName("Apache-Iceberg-Verification-Notebook") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.gcs_hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.gcs_hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.gcs_hadoop_catalog.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.gcs_hadoop_catalog.warehouse", WAREHOUSE_PATH) \
    .getOrCreate()

print("Spark Session initialized with Apache Iceberg Hadoop Catalog support!")

## Step 2: Read and Inspect current Iceberg Data

Let's load the Iceberg table written by our streaming Apache Beam pipeline (`gcs_hadoop_catalog.sensor_db.filtered_readings`) and display the schema and sample records.

In [ ]:
table_name = "gcs_hadoop_catalog.sensor_db.filtered_readings"

# Load table
df = spark.read.table(table_name)
print(f"Total records in {table_name}: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

## Step 3: Iceberg Table History and Metadata Inspection

Iceberg exposes system metadata tables directly via SQL. You can query `.history`, `.snapshots`, `.files`, or `.manifests` namespaces.

In [ ]:
print("--- Table Commit History ---")
spark.read.table(f"{table_name}.history").show(truncate=False)

print("--- Table Commit Snapshots ---")
spark.read.table(f"{table_name}.snapshots").select("committed_at", "snapshot_id", "parent_id", "operation").show(truncate=False)

print("--- Table Manifest Files ---")
spark.read.table(f"{table_name}.manifests").show(5, truncate=False)

print("--- Table Data/Metadata Files ---")
spark.read.table(f"{table_name}.files").select("file_path", "file_format", "record_count").show(5, truncate=False)

## Step 4: Demonstrate INSERT, UPDATE, and DELETE Operations

We can perform ACID DML queries directly on our GCS Iceberg table using Spark SQL. Each transaction generates a new snapshot metadata entry.

In [ ]:
# 1. Insert a new record
print("Inserting a new sensor reading record...")
spark.sql(f"INSERT INTO {table_name} VALUES ('device-101', 25.6, 60.1, 'OK', '2026-06-09T22:00:00Z')")

print("Verifying inserted record:")
spark.sql(f"SELECT * FROM {table_name} WHERE device_id = 'device-101'").show()

# 2. Update the record
print("Updating the record's humidity value...")
spark.sql(f"UPDATE {table_name} SET humidity = 62.5 WHERE device_id = 'device-101'")

print("Verifying updated record:")
spark.sql(f"SELECT * FROM {table_name} WHERE device_id = 'device-101'").show()

# 3. Delete the record
print("Deleting the inserted record...")
spark.sql(f"DELETE FROM {table_name} WHERE device_id = 'device-101'")

# 4. View snapshots after updates
print("Commit Snapshots list after DML operations:")
spark.read.table(f"{table_name}.snapshots").select("committed_at", "snapshot_id", "operation").show(truncate=False)

## Step 5: Demonstrate Time Travel

Time travel allows querying data from any past commit point using either a snapshot ID or a millisecond timestamp epoch.

In [ ]:
# Fetch a list of snapshots
snapshots = spark.read.table(f"{table_name}.snapshots").orderBy("committed_at").collect()

if len(snapshots) >= 2:
    first_snap = snapshots[0]["snapshot_id"]
    latest_snap = snapshots[-1]["snapshot_id"]
    first_snap_ts = int(snapshots[0]["committed_at"].timestamp() * 1000)
    
    # Querying by Snapshot ID
    print(f"--- Loading Data from First Snapshot ID: {first_snap} ---")
    spark.read.option("snapshot-id", first_snap).table(table_name).show(5)
    
    print(f"--- Loading Data from Latest Snapshot ID: {latest_snap} ---")
    spark.read.option("snapshot-id", latest_snap).table(table_name).show(5)
    
    # Querying by Timestamp (epoch milliseconds)
    print(f"--- Loading Data as of Timestamp: {first_snap_ts} ms ---")
    spark.read.option("as-of-timestamp", first_snap_ts).table(table_name).show(5)
else:
    print("To show time travel, keep the Beam pipeline running to generate multiple snapshots.")

## Step 6: Demonstrate Schema Evolution

Iceberg supports full schema changes (in-place column additions, renaming, type/comment updates, and dropping columns) as metadata-only actions. Old data files are not modified.

In [ ]:
# A. Add Column
print("Adding column 'operator_notes'...")
spark.sql(f"ALTER TABLE {table_name} ADD COLUMN (operator_notes string COMMENT 'Manual notes')")
spark.read.table(table_name).printSchema()

# B. Rename Column
print("Renaming column 'operator_notes' to 'notes'...")
spark.sql(f"ALTER TABLE {table_name} RENAME COLUMN operator_notes TO notes")
spark.read.table(table_name).printSchema()

# C. Alter Column Comment
print("Updating column 'notes' comment metadata...")
spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN notes COMMENT 'Updated notes comment'")

# D. Drop Column
print("Dropping column 'notes'...")
spark.sql(f"ALTER TABLE {table_name} DROP COLUMN notes")
spark.read.table(table_name).printSchema()

## Step 7: Demonstrate Partitioning and Partition Evolution

Iceberg supports in-place partition evolution. We can change the partitioning layout using SQL. The query engine reads old data using old layout specs and new data using evolved specs, avoiding any file rewrites!

In [ ]:
# A. Add a partition field
print("Evolving partition spec: adding 'status' field to partitioning layout...")
spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD status")

# B. Add a partition field using bucket transform
print("Evolving partition spec: adding 'device_id' bucket(10) transform...")
spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD bucket(10, device_id)")

# C. Drop a partition field
print("Evolving partition spec: dropping 'status' field partition layout...")
spark.sql(f"ALTER TABLE {table_name} DROP PARTITION FIELD status")

print("Partition Layout evolved successfully!")

## Step 8: Shutdown Spark Session

In [ ]:
spark.stop()
print("Spark Session shut down.")